# Pixora: ????????? ????????? ?????? ??? ???? 49 ?

???? ? ?????????????? ?????????, ??? ????? ???????? ???? 49 ? ????????? ?????????? ???????. ??? ?? ??????? ???????.

- **????????:** 16 ??????????? ???????? ?????????.
- **???????:** 160 ? ?????????? ????????? ???? ?????????, ?? ???? ??????? ?????? 10 ? ?? ?????????; ??? ????????? ?????? ???????, ? ?? ?????? ? invoice ??????????.
- **?????????? ?? ??????/????????:** ??????????? ???????? Robokassa, ?????????, ??????? ????? ?????????, ????????? ? ????????.
- **??????? ?????????:** ??? 4% ??? ??????? ???????? (????????? ???????????? ? ????????? ?????????????), ???????? 5%, ????????? 3 ? ?? ?????????, ?????? ????????? 2 ? ?? ?????????, VPS 0,50 ? ?? ????????????? ????????????.


In [1]:
from itertools import product
import pandas as pd

PRICE_RUB = 49.0
ESTIMATED_GENERATION_COST_RUB = 10.0
MEASURED_SUCCESSFUL_GENERATIONS = 16
ESTIMATED_TOTAL_GENERATION_COST_RUB = 160.0
FEE_RATE = 0.05
NPD_RATE = 0.04
SUPPORT_PER_PAYER_RUB = 3.0
REFUND_RESERVE_PER_PAYER_RUB = 2.0
VPS_PER_INVITED_USER_RUB = 0.5

def scenario(generations: int, conversion: float) -> dict:
    revenue_per_invited = PRICE_RUB * conversion
    provider_cost = generations * ESTIMATED_GENERATION_COST_RUB
    payer_variable = PRICE_RUB * (FEE_RATE + NPD_RATE) + SUPPORT_PER_PAYER_RUB + REFUND_RESERVE_PER_PAYER_RUB
    contribution_per_invited = revenue_per_invited - provider_cost - VPS_PER_INVITED_USER_RUB - payer_variable * conversion
    contribution_per_payer = contribution_per_invited / conversion
    openai_cost_per_payer = provider_cost / conversion
    return {
        'generations': generations,
        'conversion_pct': int(conversion * 100),
        'revenue_per_invited_rub': round(revenue_per_invited, 2),
        'provider_cost_per_invited_rub': round(provider_cost, 2),
        'openai_cost_per_payer_rub': round(openai_cost_per_payer, 2),
        'contribution_per_invited_rub': round(contribution_per_invited, 2),
        'contribution_per_payer_rub': round(contribution_per_payer, 2),
        'profitable': contribution_per_invited >= 0,
    }

scenarios = pd.DataFrame(scenario(g, c) for g, c in product((1, 3, 5), (0.10, 0.20, 0.30)))
scenarios

,generations,conversion_pct,revenue_per_invited_rub,provider_cost_per_invited_rub,openai_cost_per_payer_rub,contribution_per_invited_rub,contribution_per_payer_rub,profitable
0,1,10,4.9,10.0,100.00,-6.54,-65.41,False
1,1,20,9.8,10.0,50.00,-2.58,-12.91,False
2,1,30,14.7,10.0,33.33,1.38,4.59,True
3,3,10,4.9,30.0,300.00,-26.54,-265.41,False
4,3,20,9.8,30.0,150.00,-22.58,-112.91,False
5,3,30,14.7,30.0,100.00,-18.62,-62.08,False
6,5,10,4.9,50.0,500.00,-46.54,-465.41,False
7,5,20,9.8,50.0,250.00,-42.58,-212.91,False
8,5,30,14.7,50.0,166.67,-38.62,-128.74,False


In [2]:
net_per_payer_before_generation = PRICE_RUB - PRICE_RUB * (FEE_RATE + NPD_RATE) - SUPPORT_PER_PAYER_RUB - REFUND_RESERVE_PER_PAYER_RUB
break_even = pd.DataFrame([
    {
        'generations': g,
        'break_even_conversion_pct': round(100 * (g * ESTIMATED_GENERATION_COST_RUB + VPS_PER_INVITED_USER_RUB) / net_per_payer_before_generation, 1),
        'possible_at_100pct_conversion': (g * ESTIMATED_GENERATION_COST_RUB + VPS_PER_INVITED_USER_RUB) <= net_per_payer_before_generation,
    }
    for g in (1, 3, 5)
])
break_even

,generations,break_even_conversion_pct,possible_at_100pct_conversion
0,1,26.5,True
1,3,77.0,True
2,5,127.6,False


In [3]:
fee_sensitivity = pd.DataFrame([
    {
        'robokassa_fee_pct': int(rate * 100),
        'net_per_payer_before_generation_rub': round(PRICE_RUB - PRICE_RUB * (rate + NPD_RATE) - SUPPORT_PER_PAYER_RUB - REFUND_RESERVE_PER_PAYER_RUB, 2),
        'break_even_conversion_1_generation_pct': round(100 * (ESTIMATED_GENERATION_COST_RUB + VPS_PER_INVITED_USER_RUB) / (PRICE_RUB - PRICE_RUB * (rate + NPD_RATE) - SUPPORT_PER_PAYER_RUB - REFUND_RESERVE_PER_PAYER_RUB), 1),
    }
    for rate in (0.03, 0.05, 0.07)
])
fee_sensitivity

,robokassa_fee_pct,net_per_payer_before_generation_rub,break_even_conversion_1_generation_pct
0,3,40.57,25.9
1,5,39.59,26.5
2,7,38.61,27.2


## ?????????????

??? ??????? ?????????? ???? ???????? ????? ????????? 39,59 ? ?? ???????? ?? ????????? ? VPS. ???????:

- 1 ????????? ??????? ????? 26,5% ????????? ??? ??????????????;
- 3 ????????? ??????? ????? 77,0%;
- 5 ????????? ?? ????????? ???? ??? 100% ?????????.

?????????: 49 ? ???????? ?????? ??? ??????????? ???? ??? ??????? ???????? ?????????? ?????????. ?? ???????? ???????? ????? ???????? ??????????? ????????? ?? provider invoice/usage, ???????? Robokassa, ??????? ????? ????????? ? ?????????. ???? ???????????? ???????? ????? 3?5 ????????? ?? ???????, ???? ??? ?????????? ????? ???????? ??????.